# CISC3024 Pattern Recognition - AI Assignment #2

## Companion notebook 02: the ablation study

This notebook is a **standalone runner for the ablation experiment**. It exists so the
ablation can be executed without re-running the 30-epoch, four-arm main experiment in
`01_vbll_classification.ipynb`.

Everything up to the training loop is **reused byte for byte** from notebook 01: the
dependency self-repair, the configuration, the environment guards, the data, the shared
backbone, the VBLL head factory, the `Prediction` contract, the adapters, the metrics,
`evaluate_model` and `train_model`. Only the tail differs - here we run the six ablations
and nothing else.

What this notebook deliberately does **not** do: it never trains the four main arms, never
fits a Laplace posterior, and never reproduces the main figures. Those belong to notebook 01.

**Expected output:** `outputs/results_ablations.csv` (six rows) and a Markdown table ready
to paste into report section 4.4.

## Section 0 - environment and configuration

In [1]:
# =============================================================================
# Section 0 - dependency self-repair (hotfix for a recycled Colab session)
# =============================================================================
# Symptom this fixes
#     ModuleNotFoundError: No module named 'vbll'
#     raised by the "environment echo and fail-fast guards" cell further down.
#
# Cause
#     Colab discards every pip-installed package when a session is recycled or
#     the runtime disconnects. 00_colab_setup.ipynb did its job -- in a session
#     that no longer exists. The packages are gone, not broken.
#
# What this cell does
#     1. reports what is importable right now;
#     2. records Colab's OWN torch / torchvision versions into a constraints file;
#     3. installs vbll + laplace-torch (and the plain scientific stack) BEHIND
#        that constraint, so no transitive dependency can swap the CUDA build of
#        PyTorch for a CPU wheel and silently make cuda.is_available() False;
#     4. confirms torch.cuda.is_available() is still True afterwards.
#
# It is idempotent: run it as often as you like. If everything is already
# importable it only reports and exits without touching the environment, so it
# costs a couple of seconds on a healthy session.
# =============================================================================
import importlib
import pathlib
import platform
import subprocess
import sys
import warnings

# Importing laplace pulls in torch.jit users that emit deprecation warnings on
# modern torch. The notebook ignores warnings globally in its config cell; do the
# same here so this cell's output stays readable.
warnings.filterwarnings("ignore")

_NL = chr(10)


def _rule(title):
    print("=" * 68)
    print(title)
    print("=" * 68)


def _version(module_name):
    """Installed __version__ of a module, or None if it cannot be imported."""
    try:
        module = importlib.import_module(module_name)
    except Exception:
        return None
    return getattr(module, "__version__", "n/a")


def _base(version):
    """'2.6.0+cu124' -> '2.6.0'  (strip the local build tag)."""
    return str(version).split("+")[0]


# ---------------------------------------------------------------------------
# Step 0 - PyTorch must already exist. Colab always ships it; if it is missing
# the runtime is broken and no amount of pip will help.
# ---------------------------------------------------------------------------
try:
    import torch
    import torchvision
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "PyTorch itself is not importable ({}). The Colab runtime is broken, not "
        "merely missing packages. Use Runtime > Restart session, then run this "
        "cell again.".format(exc)
    ) from None

REQUIRED = ["torch", "torchvision", "numpy", "scipy", "pandas", "sklearn",
            "matplotlib", "seaborn", "tqdm", "vbll", "laplace"]

_rule("HOTFIX 1/4 - current state of the runtime")
print("  python       :", platform.python_version())
print("  torch        :", torch.__version__)
print("  torchvision  :", torchvision.__version__)
print("  CUDA runtime :", torch.version.cuda)
print("  GPU          :",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print()

_MISSING = [name for name in REQUIRED if _version(name) is None]
print("  missing      :", ", ".join(_MISSING) if _MISSING else "nothing")

# ---------------------------------------------------------------------------
# Step 1 - pin torch / torchvision to exactly what Colab already has
# ---------------------------------------------------------------------------
_rule("HOTFIX 2/4 - write the constraints file")
_CONSTRAINTS = pathlib.Path("colab-constraints.txt")
_CONSTRAINTS.write_text(_NL.join([
    "torch==" + _base(torch.__version__),
    "torchvision==" + _base(torchvision.__version__),
]) + _NL)

print("  wrote", _CONSTRAINTS.resolve())
print()
for _line in _CONSTRAINTS.read_text().rstrip().split(_NL):
    print("      " + _line)
print()
print("  ^ this is what stops pip replacing Colab's CUDA build of torch.")

# ---------------------------------------------------------------------------
# Step 2 - install everything else behind that constraint
# ---------------------------------------------------------------------------
_rule("HOTFIX 3/4 - install behind the constraint")

if not _MISSING:
    print("  nothing missing - skipping the install (this cell is idempotent).")
else:
    _PACKAGES = [
        "vbll==0.4.9",           # the algorithm under study (ICLR 2024)
        "laplace-torch==0.2.3",  # comparison baseline (last-layer Laplace)
        "numpy", "scipy", "pandas", "scikit-learn",
        "matplotlib", "seaborn", "tqdm",
    ]
    _cmd = [sys.executable, "-m", "pip", "install", "-q",
            "-c", str(_CONSTRAINTS), *_PACKAGES]
    print("  $ " + " ".join(_cmd))
    print()
    try:
        subprocess.run(_cmd, check=True)
    except subprocess.CalledProcessError as _exc:
        print()
        print("  INSTALL FAILED (exit code {}).".format(_exc.returncode))
        print("  Diagnose with these, pasted into a new cell:")
        print("      !{} -m pip check".format(sys.executable))
        print("      !{} -m pip install -c colab-constraints.txt "
              "vbll==0.4.9 laplace-torch==0.2.3 -v".format(sys.executable))
        raise

    # A running kernel caches the contents of site-packages, so a freshly
    # installed top-level package is not visible until those caches are dropped.
    importlib.invalidate_caches()
    sys.path_importer_cache.clear()
    for _name in [n for n in list(sys.modules)
                  if n.split(".")[0] in ("vbll", "laplace")]:
        del sys.modules[_name]
    print()
    print("  install finished; import caches dropped")

# ---------------------------------------------------------------------------
# Step 3 - verify: every import works AND torch was not disturbed
# ---------------------------------------------------------------------------
_rule("HOTFIX 4/4 - verify")
importlib.invalidate_caches()

print("  torch        :", torch.__version__, "   <- must be unchanged")
print("  CUDA runtime :", torch.version.cuda)
print()
print("  {:<12} {:<16} {}".format("module", "version", "status"))
print("  " + "-" * 42)

_still_missing = []
for _name in REQUIRED:
    _v = _version(_name)
    if _v is None:
        _still_missing.append(_name)
        print("  {:<12} {:<16} MISSING".format(_name, "-"))
    else:
        print("  {:<12} {:<16} OK".format(_name, _v))

print()
if _still_missing:
    print("  STILL MISSING:", ", ".join(_still_missing))
    print()
    print("  Python caches site-packages contents, so a package installed while")
    print("  the kernel is already running is sometimes not visible until the")
    print("  runtime restarts. Use Runtime > Restart session, then run this cell")
    print("  and Runtime > Run all.")
    raise RuntimeError("hotfix incomplete: {}".format(_still_missing))

if not torch.cuda.is_available():
    print("  !! torch.cuda.is_available() is False.")
    print("     The install did NOT break torch (the version above is unchanged),")
    print("     so the GPU was never attached. Fix it with:")
    print("       Runtime > Change runtime type > T4 GPU > Save")
    print("       Runtime > Restart session, then Runtime > Run all")
    print()
    print("  The environment is otherwise ready, so you can also continue on CPU")
    print("  by setting CFG['require_cuda'] = False in the config cell below.")
else:
    print("  GPU          :", torch.cuda.get_device_name(0))
    print()
    print("  HOTFIX OK - environment repaired, torch untouched, CUDA alive.")
    print("  Continue with the cells below (Runtime > Run all if you restarted).")

HOTFIX 1/4 - current state of the runtime
  python       : 3.13.15
  torch        : 2.11.0+cu130
  torchvision  : 0.26.0+cu130
  CUDA runtime : 13.0
  GPU          : Tesla T4

  missing      : vbll, laplace
HOTFIX 2/4 - write the constraints file
  wrote /content/colab-constraints.txt

      torch==2.11.0
      torchvision==0.26.0

  ^ this is what stops pip replacing Colab's CUDA build of torch.
HOTFIX 3/4 - install behind the constraint
  $ /usr/bin/python3 -m pip install -q -c colab-constraints.txt vbll==0.4.9 laplace-torch==0.2.3 numpy scipy pandas scikit-learn matplotlib seaborn tqdm


  install finished; import caches dropped
HOTFIX 4/4 - verify
  torch        : 2.11.0+cu130    <- must be unchanged
  CUDA runtime : 13.0

  module       version          status
  ------------------------------------------
  torch        2.11.0+cu130     OK
  torchvision  0.26.0+cu130     OK
  numpy        2.1.3            OK
  scipy        1.16.3           OK
  pandas       2.2.3            OK
  sk

In [2]:
# =============================================================================
# Section 0 - imports, global configuration, seeding
# =============================================================================
import os
import json
import time
import random
import warnings
import importlib.metadata as _md
from dataclasses import dataclass, field
from typing import Optional, Dict, Any, List, Tuple, Callable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# Device and global seed
# ----------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 0

# ----------------------------------------------------------------------------
# A single CFG mapping - every tunable lives here, so the report can quote it
# verbatim and an ablation is a one-line change.
# ----------------------------------------------------------------------------
CFG: Dict[str, Any] = {
    # reproducibility
    "seed": SEED,
    "seeds": [0, 1, 2],            # used by Section 6 only
    "require_cuda": True,          # fail fast if the Colab GPU is not enabled

    # data
    "data_dir": "./data",
    "val_frac": 0.10,
    "num_workers": 2,

    # shared model
    "in_dim": 28 * 28,
    "hidden": 256,
    "depth": 2,
    "num_classes": 10,

    # optimisation (identical for every arm)
    "epochs": 30,
    "batch_size": 512,
    "lr": 3e-3,
    "weight_decay": 0.0,           # must stay 0 for the VBLL head; see train_model
    "early_stop_patience": 8,

    # VBLL head
    "vbll_parameterization": "diagonal",   # {'diagonal', 'dense', 'lowrank'}
    "vbll_prior_scale": 1.0,
    "vbll_return_ood": True,               # the adapter REQUIRES this to be True

    # Laplace (arm d)
    "laplace_subset": "last_layer",
    "laplace_hessian": "kron",
    "laplace_pred_type": "glm",
    "laplace_link_approx": "sampled",      # 'sampled' -> 'mc' | 'deterministic' -> 'probit'
    "laplace_n_samples": 100,              # only used by link_approx='mc'
    "laplace_grid_size": 30,
    "laplace_log_prior_min": -4.0,
    "laplace_log_prior_max": 4.0,
    "laplace_batch_size": 512,             # batch the GLM predictive to avoid OOM
    "laplace_diagonal_output": False,

    # metrics
    "n_bins": 15,                          # equal-width ECE bins

    # outputs
    "out_dir": "outputs",
    "fig_dir": "outputs/figures",

    # Section 6 gate
    "run_robustness": False,
}

# The human-readable mode names mapped onto the values laplace-torch accepts.
# laplace-torch validates link_approx against {'mc', 'probit', 'bridge', 'bridge_norm'}.
LINK_APPROX_MAP = {"sampled": "mc", "deterministic": "probit"}

os.makedirs(CFG["out_dir"], exist_ok=True)
os.makedirs(CFG["fig_dir"], exist_ok=True)


def set_seed(seed: int) -> None:
    """Seed python, numpy and torch (CPU + CUDA) so a run is repeatable."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CFG["seed"])

print("device      :", DEVICE)
print("torch       :", torch.__version__)
print("numpy       :", np.__version__)
print("output dir  :", os.path.abspath(CFG["out_dir"]))

device      : cuda
torch       : 2.11.0+cu130
numpy       : 2.1.3
output dir  : /content/outputs


In [3]:
# =============================================================================
# Ablation run configuration
# =============================================================================
# In notebook 01 the ablation hooks sit behind the CFG["run_robustness"] gate, which is
# OFF by default. Here we turn the gate on explicitly and pin a single seed: the ablation
# varies ONE factor at a time, so a multi-seed loop would multiply the cost without
# answering a different question. (The multi-seed cell is not present in this notebook,
# so CFG["seeds"] is set to [0] purely as a safety net.)
CFG["run_robustness"] = True
CFG["seeds"] = [0]

print("run_robustness :", CFG["run_robustness"])
print("seeds          :", CFG["seeds"])
print("epochs per arm :", CFG["epochs"])
print("arms           : 6")
print()
print("Six arms at {} epochs each. 'disc_dense' is by far the most expensive;".format(CFG["epochs"]))
print("see the note below the Section 6 header before starting.")

run_robustness : True
seeds          : [0]
epochs per arm : 30
arms           : 6

Six arms at 30 epochs each. 'disc_dense' is by far the most expensive;
see the note below the Section 6 header before starting.


In [4]:
# =============================================================================
# Section 0 - environment echo and fail-fast guards
# =============================================================================
def pkg_version(name: str) -> str:
    """Installed version of a distribution, or 'NOT-INSTALLED'."""
    try:
        return _md.version(name)
    except _md.PackageNotFoundError:
        return "NOT-INSTALLED"


# Import the two libraries the whole notebook depends on. If the setup notebook
# was not run, this fails here rather than fifteen cells later.
import vbll
from laplace import Laplace

ENV_INFO: Dict[str, Any] = {
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "numpy": np.__version__,
    "vbll": pkg_version("vbll"),
    "laplace-torch": pkg_version("laplace-torch"),
    "curvlinops-for-pytorch": pkg_version("curvlinops-for-pytorch"),
    "scikit-learn": pkg_version("scikit-learn"),
}

print("environment echo")
print("-" * 52)
for _k, _v in ENV_INFO.items():
    print("  {:<24} {}".format(_k, _v))

if CFG["require_cuda"] and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. In Colab use Runtime > Change runtime type > "
        "T4 GPU and re-run. To deliberately run on CPU instead, set "
        "CFG['require_cuda'] = False."
    )

print()
print("guard passed: vbll and laplace-torch both import cleanly")

environment echo
----------------------------------------------------
  device                   cuda
  gpu_name                 Tesla T4
  torch                    2.11.0+cu130
  torchvision              0.26.0+cu130
  numpy                    2.1.3
  vbll                     0.4.9
  laplace-torch            0.2.3
  curvlinops-for-pytorch   3.0.1
  scikit-learn             1.6.1

guard passed: vbll and laplace-torch both import cleanly


## Section 1 - data

In [5]:
# =============================================================================
# Section 1 - data
# =============================================================================
MEAN, STD = (0.1307,), (0.3081,)

train_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_full = datasets.MNIST(CFG["data_dir"], train=True, download=True, transform=train_tf)
test_ds = datasets.MNIST(CFG["data_dir"], train=False, download=True, transform=eval_tf)
ood_ds = datasets.FashionMNIST(CFG["data_dir"], train=False, download=True, transform=eval_tf)

n_val = int(len(train_full) * CFG["val_frac"])
n_train = len(train_full) - n_val

# The split is carved ONCE, from a fixed-seed generator. Every arm and every seed
# run therefore trains and validates on exactly the same samples.
_split_gen = torch.Generator().manual_seed(CFG["seed"])
train_ds, val_ds = random_split(train_full, [n_train, n_val], generator=_split_gen)


def make_loader(ds, shuffle: bool, batch_size: Optional[int] = None) -> DataLoader:
    return DataLoader(
        ds,
        batch_size=batch_size if batch_size is not None else CFG["batch_size"],
        shuffle=shuffle,
        num_workers=CFG["num_workers"],
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )


train_loader = make_loader(train_ds, shuffle=True)
val_loader = make_loader(val_ds, shuffle=False)
test_loader = make_loader(test_ds, shuffle=False)
ood_loader = make_loader(ood_ds, shuffle=False)

print("train      {:>6d}".format(len(train_ds)))
print("val        {:>6d}".format(len(val_ds)))
print("test (ID)  {:>6d}".format(len(test_ds)))
print("OOD        {:>6d}".format(len(ood_ds)))

assert len(train_full) == 60000, "MNIST train split size changed"
assert len(train_ds) + len(val_ds) == len(train_full), "train/val split lost samples"
assert len(test_ds) == 10000, "MNIST test split size changed"
assert len(ood_ds) == 10000, "Fashion-MNIST test split size changed"
print("data guard: all four dataset sizes are as expected")

100%|██████████| 9.91M/9.91M [00:13<00:00, 720kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 128kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.22MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.8MB/s]
100%|██████████| 26.4M/26.4M [00:02<00:00, 10.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 169kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.18MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 15.1MB/s]

train       54000
val          6000
test (ID)   10000
OOD         10000
data guard: all four dataset sizes are as expected


## Section 2 - the shared model and the VBLL heads

In [6]:
# =============================================================================
# Section 2 - the shared backbone
# =============================================================================
class MLPBackbone(nn.Module):
    """Shared MLP trunk: `depth` hidden layers of `hidden` units with ReLU."""

    def __init__(self, in_dim: int = 784, hidden: int = 256, depth: int = 2):
        super().__init__()
        layers: List[nn.Module] = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.ReLU()]
            d = hidden
        self.features = nn.Sequential(*layers)
        self.out_dim = d

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)
        return self.features(x)


def build_backbone() -> MLPBackbone:
    return MLPBackbone(CFG["in_dim"], CFG["hidden"], CFG["depth"])


def count_params(model: nn.Module) -> int:
    """Trainable parameter count - a report artefact for the complexity comparison."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


_bb = build_backbone()
BACKBONE_PARAMS = count_params(_bb)
print("backbone output dim :", _bb.out_dim)
print("backbone params     :", BACKBONE_PARAMS)
del _bb

backbone output dim : 256
backbone params     : 266752


In [7]:
# =============================================================================
# Section 2 - arms (b) and (c): VBLL heads
# =============================================================================
# -----------------------------------------------------------------------------
# Device-safety shim for vbll 0.4.9 -- an upstream bug, not our code
# -----------------------------------------------------------------------------
# vbll allocates two helper identity matrices WITHOUT a `device=` argument, so they
# are created on the CPU while the model lives on the GPU:
#
#   vbll/utils/distributions.py:327
#       term2 = torch.linalg.det(arg1 + torch.eye(arg1.shape[-1])).log()
#       reached by LowRankNormal.logdet_covariance, i.e. parameterization='lowrank'
#   vbll/utils/distributions.py:27
#       torch.cholesky_solve(torch.eye(u.size(-1)).expand(u.size()), u, upper=upper)
#       reached by cholesky_inverse(), used by the precision parameterizations
#
# On a GPU run line 327 raises:
#   RuntimeError: Expected all tensors to be on the same device, but found at least
#   two devices, cuda:0 and cpu!
#
# The 'diagonal' parameterization never touches either line, which is why the main
# run (parameterization='diagonal') was unaffected and only the lowrank ablation arm
# failed. Editing site-packages is not an option: Colab discards it on every session
# recycle. Both functions are therefore replaced once, here, with equivalents that
# differ only in that the identity matrix inherits device and dtype from the tensor
# it is combined with. The math is otherwise identical to the library's.
import vbll.utils.distributions as _vbll_dist


def _lowrank_logdet_covariance(self):
    term1 = torch.log(self.cov_diag).sum(-1)
    arg1 = _vbll_dist.tp(self.cov_factor) @ (self.cov_factor / self.cov_diag.unsqueeze(-1))
    eye = torch.eye(arg1.shape[-1], device=arg1.device, dtype=arg1.dtype)
    return term1 + torch.linalg.det(arg1 + eye).log()


def _cholesky_inverse(u, upper=False):
    if u.dim() == 2 and not u.requires_grad:
        return torch.cholesky_inverse(u, upper=upper)
    eye = torch.eye(u.size(-1), device=u.device, dtype=u.dtype).expand(u.size())
    return torch.cholesky_solve(eye, u, upper=upper)


_vbll_dist.LowRankNormal.logdet_covariance = property(_lowrank_logdet_covariance)
_vbll_dist.cholesky_inverse = _cholesky_inverse

print("vbll device-safety shim installed (lowrank logdet + cholesky_inverse)")


class VBLLModel(nn.Module):
    """Backbone + VBLL head.

    forward() returns the library's VBLL output object, not a tensor. That object
    carries the predictive distribution, the training loss function, the validation
    loss function and (when return_ood=True) the OOD scores. The adapter is the only
    code that unpacks it.
    """

    def __init__(self, head: nn.Module):
        super().__init__()
        self.backbone = build_backbone()
        self.head = head

    def forward(self, x: torch.Tensor):
        return self.head(self.backbone(x))


def build_vbll_model(
    kind: str = "disc",
    reg_weight: Optional[float] = None,
    parameterization: Optional[str] = None,
    prior_scale: Optional[float] = None,
    return_ood: Optional[bool] = None,
) -> VBLLModel:
    """Factory for arms (b) and (c).

    kind='disc' -> vbll.DiscClassification  (discriminative, Jensen softmax bound)
    kind='gen'  -> vbll.GenClassification   (generative, class-conditional likelihood)

    Defaults come from CFG, and `return_ood` defaults to True because the adapter
    depends on `out.ood_scores` existing.
    """
    if parameterization is None:
        parameterization = CFG["vbll_parameterization"]
    if prior_scale is None:
        prior_scale = CFG["vbll_prior_scale"]
    if return_ood is None:
        return_ood = CFG["vbll_return_ood"]
    if reg_weight is None:
        # The ELBO regularisation term is scaled by 1/N so that it stays
        # comparable with the data term as the dataset grows.
        reg_weight = 1.0 / len(train_ds)

    in_features = CFG["hidden"]

    if kind == "disc":
        head = vbll.DiscClassification(
            in_features=in_features,
            out_features=CFG["num_classes"],
            regularization_weight=reg_weight,
            parameterization=parameterization,
            return_ood=return_ood,
            prior_scale=prior_scale,
        )
    elif kind == "gen":
        head = vbll.GenClassification(
            in_features=in_features,
            out_features=CFG["num_classes"],
            regularization_weight=reg_weight,
            parameterization=parameterization,
            return_ood=return_ood,
            prior_scale=prior_scale,
        )
    else:
        raise ValueError("kind must be 'disc' or 'gen', got {!r}".format(kind))

    return VBLLModel(head).to(DEVICE)


print("arm (b) total params:", count_params(build_vbll_model("disc")))
print("arm (c) total params:", count_params(build_vbll_model("gen")))
print("VBLL regularisation weight (1/N_train):", 1.0 / len(train_ds))

vbll device-safety shim installed (lowrank logdet + cholesky_inverse)
arm (b) total params: 271882
arm (c) total params: 272128
VBLL regularisation weight (1/N_train): 1.8518518518518518e-05


## Section 3 - the evaluation contract

In [8]:
# =============================================================================
# Section 3 - the Prediction container and its validator
# =============================================================================
from contextlib import contextmanager, nullcontext


@contextmanager
def deterministic_mc(seed: int):
    """Pin the RNG for the duration of a VBLL predictive call.

    VBLL's `predictive()` averages a FIXED number of internal Monte-Carlo samples
    (20, hard-coded in the library), so its output is stochastic even though the
    method itself is sampling-free. Pinning the RNG here
      * makes the reported numbers reproducible run to run, and
      * gives the ID pass and the OOD pass the same noise pattern.
    The caller's RNG state is restored afterwards, so training is unaffected.
    """
    state = torch.get_rng_state()
    cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    torch.manual_seed(seed)
    try:
        yield
    finally:
        torch.set_rng_state(state)
        if cuda_state is not None:
            torch.cuda.set_rng_state_all(cuda_state)


@dataclass
class Prediction:
    """The only thing evaluate_model ever consumes."""

    probs: np.ndarray
    logits: Optional[np.ndarray] = None
    ood_score: Optional[np.ndarray] = None          # higher == more OOD
    uncertainty_aleatoric: Optional[np.ndarray] = None
    uncertainty_epistemic: Optional[np.ndarray] = None
    extra: Dict[str, Any] = field(default_factory=dict)

    @property
    def n(self) -> int:
        return int(self.probs.shape[0])


def check_contract(pred: Prediction, arm_name: str, expected_n: Optional[int] = None,
                   tol: float = 1e-4) -> bool:
    """Assert every invariant of the Prediction contract.

    Called by each adapter on its OWN output, so a violation names the arm that
    caused it instead of surfacing later inside the metrics.
    """
    if not isinstance(pred, Prediction):
        raise TypeError("[{}] predict() must return a Prediction, got {}".format(
            arm_name, type(pred).__name__))

    p = np.asarray(pred.probs, dtype=np.float64)
    if p.ndim != 2:
        raise ValueError("[{}] probs must be 2-D (N, C); got shape {}".format(arm_name, p.shape))
    n, c = p.shape

    if expected_n is not None and n != expected_n:
        raise ValueError("[{}] probs has N={}, but the dataset has {}".format(
            arm_name, n, expected_n))
    if c != CFG["num_classes"]:
        raise ValueError("[{}] probs has C={}, expected {}".format(
            arm_name, c, CFG["num_classes"]))
    if not np.isfinite(p).all():
        raise ValueError("[{}] probs contains NaN or inf".format(arm_name))
    if p.min() < -1e-6 or p.max() > 1.0 + 1e-6:
        raise ValueError("[{}] probs outside [0, 1] (min={:.6g}, max={:.6g})".format(
            arm_name, p.min(), p.max()))

    row_sum = p.sum(axis=1)
    if not np.allclose(row_sum, 1.0, atol=tol):
        raise ValueError("[{}] probs rows do not sum to 1 "
                         "(min={:.8f}, max={:.8f})".format(arm_name, row_sum.min(), row_sum.max()))

    if pred.ood_score is not None:
        s = np.asarray(pred.ood_score, dtype=np.float64)
        if s.shape != (n,):
            raise ValueError("[{}] ood_score must have shape ({},); got {}".format(
                arm_name, n, s.shape))
        if not np.isfinite(s).all():
            raise ValueError("[{}] ood_score contains NaN or inf".format(arm_name))

    for _fld in ("uncertainty_aleatoric", "uncertainty_epistemic"):
        _v = getattr(pred, _fld)
        if _v is not None and np.asarray(_v).shape != (n,):
            raise ValueError("[{}] {} must have shape ({},); got {}".format(
                arm_name, _fld, n, np.asarray(_v).shape))

    return True


def collect_targets(loader: DataLoader) -> np.ndarray:
    """Labels of a loader, in the same order the adapter concatenated its batches.

    The loaders yield CPU tensors, so this is already device-safe; the `.cpu()` is
    defensive so the helper stays correct if a loader is ever changed to yield
    device tensors.
    """
    ys = []
    for _, y in loader:
        if isinstance(y, torch.Tensor):
            y = y.detach().cpu()
        ys.append(np.asarray(y))
    return np.concatenate(ys, axis=0)


def predictive_entropy(probs: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """H[p] in nats. Defined for every arm, which is what makes it the fair score."""
    p = np.clip(np.asarray(probs, dtype=np.float64), eps, 1.0)
    return -(p * np.log(p)).sum(axis=1)

In [9]:
# =============================================================================
# Section 3 - the three adapters
# =============================================================================
class BaselineAdapter:
    """Arm (a). Raw logits -> softmax, once. ood_score = 1 - max p."""

    def __init__(self, model: nn.Module, name: str = "a_mlp_softmax"):
        self.model = model
        self.name = name

    @torch.no_grad()
    def predict(self, loader: DataLoader) -> Prediction:
        self.model.eval()
        logits_chunks, prob_chunks = [], []
        for x, _ in loader:
            z = self.model(x.to(DEVICE))
            logits_chunks.append(z.detach().cpu().numpy())
            prob_chunks.append(F.softmax(z, dim=-1).detach().cpu().numpy())

        logits = np.concatenate(logits_chunks, axis=0).astype(np.float64)
        probs = np.concatenate(prob_chunks, axis=0).astype(np.float64)

        pred = Prediction(
            probs=probs,
            logits=logits,
            ood_score=(1.0 - probs.max(axis=1)).astype(np.float64),
            extra={"source": "softmax", "ood_score_definition": "1 - max p"},
        )
        check_contract(pred, self.name, expected_n=len(loader.dataset))
        return pred


class VBLLAdapter:
    """Arms (b) and (c). Reads the VBLL output object directly.

    Two things deliberately do NOT happen here:
      * no softmax - the VBLL head already returns normalised probabilities;
      * no Monte-Carlo loop - VBLL's predictive is sampling-free by design.
    """

    def __init__(self, model: nn.Module, name: str):
        self.model = model
        self.name = name

    @torch.no_grad()
    def predict(self, loader: DataLoader) -> Prediction:
        self.model.eval()
        prob_chunks, ood_chunks = [], []

        # vbll's predictive() is stochastic (it averages 20 internal MC samples),
        # so pin the RNG to make the reported numbers reproducible.
        with deterministic_mc(CFG["seed"]):
            for x, _ in loader:
                x = x.to(DEVICE)
                out = self.model(x)

                # Fail loudly and by name if the head was built without return_ood.
                raw_ood = getattr(out, "ood_scores", None)
                if raw_ood is None:
                    raise RuntimeError(
                        "[{}] the VBLL head was built with return_ood=False, so "
                        "out.ood_scores does not exist. Rebuild it with "
                        "CFG['vbll_return_ood'] = True.".format(self.name))
                if callable(raw_ood):
                    raw_ood = raw_ood(x)

                # out.predictive is already a normalised Categorical -> read .probs.
                prob_chunks.append(out.predictive.probs.detach().cpu().numpy())
                ood_chunks.append(raw_ood.detach().cpu().numpy().reshape(-1))

        probs = np.concatenate(prob_chunks, axis=0).astype(np.float64)
        ood_raw = np.concatenate(ood_chunks, axis=0).astype(np.float64)

        # vbll defines ood_scores as the MAX PREDICTIVE PROBABILITY:
        # a large value means "confident", i.e. in-distribution. Flip it so that
        # the Prediction contract (higher == more OOD) holds for every arm.
        ood_score = (1.0 - ood_raw).astype(np.float64)

        pred = Prediction(
            probs=probs,
            logits=np.log(np.clip(probs, 1e-12, 1.0)),
            ood_score=ood_score,
            extra={
                "source": "vbll",
                "vbll_ood_raw_definition": "max predictive probability",
                "ood_score_transform": "1 - max_predictive",
                "mc_samples_inside_head": 20,
            },
        )
        check_contract(pred, self.name, expected_n=len(loader.dataset))
        return pred


class LaplaceAdapter:
    """Arm (d). laplace-torch returns normalised probabilities from its GLM
    predictive, so again no softmax is applied here.

    laplace-torch has NO native OOD score. It is therefore DERIVED from the
    predictive distribution. That is a genuine methodological asymmetry with the
    VBLL arms, which is exactly why the notebook also reports the common entropy
    score family.
    """

    def __init__(self, la, name: str = "d_laplace_lla", link_approx: str = "probit",
                 n_samples: int = 100, batch_size: int = 512,
                 diagonal_output: bool = False):
        self.la = la
        self.name = name
        self.link_approx = link_approx
        self.n_samples = n_samples
        self.batch_size = batch_size
        self.diagonal_output = diagonal_output

    @torch.no_grad()
    def predict(self, loader: DataLoader) -> Prediction:
        self.la.model.eval()
        outer = []

        for x, _ in loader:
            x = x.to(DEVICE)
            # The GLM predictive is the heaviest step in the notebook: batch it so
            # a large test set cannot exhaust memory.
            inner = []
            for i in range(0, x.shape[0], self.batch_size):
                xb = x[i:i + self.batch_size]
                p = self.la(
                    xb,
                    pred_type=CFG["laplace_pred_type"],
                    link_approx=self.link_approx,
                    n_samples=self.n_samples,
                    diagonal_output=self.diagonal_output,
                )
                inner.append(p.detach().cpu().numpy())
            outer.append(np.concatenate(inner, axis=0))

        probs = np.concatenate(outer, axis=0).astype(np.float64)
        # Numerical guard only - the link approximations are already normalised.
        probs = probs / probs.sum(axis=1, keepdims=True)

        pred = Prediction(
            probs=probs,
            logits=np.log(np.clip(probs, 1e-12, 1.0)),
            ood_score=(1.0 - probs.max(axis=1)).astype(np.float64),
            extra={
                "source": "laplace_glm",
                "link_approx": self.link_approx,
                "n_samples": self.n_samples,
                "hessian_structure": CFG["laplace_hessian"],
                "ood_score_definition": "derived: 1 - max p",
            },
        )
        check_contract(pred, self.name, expected_n=len(loader.dataset))
        return pred

In [10]:
# =============================================================================
# Section 3 - metrics: pure functions of arrays
# =============================================================================
# No model, no loader, no device. That makes each one independently checkable,
# which matters because a wrong ECE silently invalidates a headline claim.
def accuracy(probs: np.ndarray, y: np.ndarray) -> float:
    probs = np.asarray(probs)
    y = np.asarray(y)
    return float((probs.argmax(axis=1) == y).mean())


def nll(probs: np.ndarray, y: np.ndarray, eps: float = 1e-12) -> float:
    probs = np.asarray(probs, dtype=np.float64)
    y = np.asarray(y)
    p_true = np.clip(probs[np.arange(len(y)), y], eps, 1.0)
    return float(-np.log(p_true).mean())


def ece(probs: np.ndarray, y: np.ndarray, n_bins: int = 15):
    """Expected Calibration Error with EQUAL-WIDTH bins.

    Returns the scalar ECE *plus* the bin arrays, so the reliability diagram is a
    by-product of the metric and can never drift out of sync with the number
    printed in the results table.

    Returns
    -------
    (ece_value, bin_conf, bin_acc, bin_count, bin_edges)
    """
    probs = np.asarray(probs, dtype=np.float64)
    y = np.asarray(y)

    conf = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == y).astype(np.float64)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    # Interior edges only; np.digitize then yields 0..n_bins, clipped to n_bins-1
    # so that confidence == 1.0 lands in the last bin rather than falling off.
    bin_ids = np.clip(np.digitize(conf, edges[1:-1], right=False), 0, n_bins - 1)

    bin_count = np.zeros(n_bins, dtype=np.float64)
    bin_conf = np.zeros(n_bins, dtype=np.float64)
    bin_acc = np.zeros(n_bins, dtype=np.float64)

    for b in range(n_bins):
        m = bin_ids == b
        cnt = int(m.sum())
        bin_count[b] = cnt
        if cnt > 0:
            bin_conf[b] = conf[m].mean()
            bin_acc[b] = correct[m].mean()

    total = float(len(conf))
    ece_value = float(np.sum(bin_count / total * np.abs(bin_acc - bin_conf)))
    return ece_value, bin_conf, bin_acc, bin_count, edges


def ece_equal_mass(probs: np.ndarray, y: np.ndarray, n_bins: int = 15) -> float:
    """Expected Calibration Error with EQUAL-MASS (adaptive) bins.

    The standard alternative to the equal-width default, and the reason it matters:
    on a saturated benchmark such as MNIST almost every test sample lands in the
    top equal-width bin, so the equal-width estimator is effectively dominated by
    that one bin. Equal-mass binning splits the samples into `n_bins` groups of
    roughly equal size, which weights the low-confidence region far more heavily.

    Reporting both makes the sensitivity of the headline ECE to the binning choice
    explicit instead of leaving it hidden. Equal-width remains the default because
    it is the more common convention.

    Caveat: equal-mass binning needs n_bins << N. If n_bins approaches N every bin
    holds a single sample, whose "empirical accuracy" is 0 or 1, and the estimator
    degenerates. With N = 10000 and n_bins = 15 that is far from a concern.
    """
    probs = np.asarray(probs, dtype=np.float64)
    y = np.asarray(y)

    conf = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == y).astype(np.float64)

    order = np.argsort(conf)
    conf_sorted = conf[order]
    correct_sorted = correct[order]

    groups = np.array_split(np.arange(len(conf_sorted)), n_bins)
    total = float(len(conf_sorted))

    ece_value = 0.0
    for g in groups:
        if len(g) == 0:
            continue
        ece_value += (len(g) / total) * abs(correct_sorted[g].mean() - conf_sorted[g].mean())
    return float(ece_value)


def auroc(scores_id: np.ndarray, scores_ood: np.ndarray) -> float:
    """AUROC with the OOD samples as the positive class.

    Inputs follow the Prediction convention: higher score == more OOD, so a healthy
    detector scores above 0.5. An arm that returns a consistently inverted score
    yields AUROC < 0.5 - that is a signal, and it is deliberately kept visible
    instead of being auto-flipped.
    """
    s_id = np.asarray(scores_id, dtype=np.float64)
    s_ood = np.asarray(scores_ood, dtype=np.float64)
    y = np.concatenate([np.zeros(len(s_id)), np.ones(len(s_ood))])
    s = np.concatenate([s_id, s_ood])
    return float(roc_auc_score(y, s))

In [11]:
# =============================================================================
# Section 3 - evaluate_model: the single entry point
# =============================================================================
def evaluate_model(adapter, id_loader: DataLoader, ood_loader: DataLoader,
                   n_bins: Optional[int] = None) -> Dict[str, Any]:
    """Run one adapter over the ID and OOD loaders and return every number and
    every array the report needs.

    This function never inspects a model type. It only ever touches Predictions.
    """
    n_bins = CFG["n_bins"] if n_bins is None else n_bins

    pred_id = adapter.predict(id_loader)
    pred_ood = adapter.predict(ood_loader)

    # Re-validate at the boundary too: cheap, and it means a contract violation can
    # never leak into the metrics.
    check_contract(pred_id, adapter.name, expected_n=len(id_loader.dataset))
    check_contract(pred_ood, adapter.name, expected_n=len(ood_loader.dataset))

    y_id = collect_targets(id_loader)

    ece_value, bin_conf, bin_acc, bin_count, bin_edges = ece(pred_id.probs, y_id, n_bins=n_bins)

    # The same metric under the other binning convention. On a saturated benchmark
    # the two can differ by a lot, so both are reported rather than one being
    # chosen silently.
    ece_em = ece_equal_mass(pred_id.probs, y_id, n_bins=n_bins)

    # family 1 - each arm's own score (this is what the method claims)
    auroc_native = auroc(pred_id.ood_score, pred_ood.ood_score)

    # family 2 - predictive entropy, computable for ALL four arms (fair comparison)
    ent_id = predictive_entropy(pred_id.probs)
    ent_ood = predictive_entropy(pred_ood.probs)
    auroc_entropy = auroc(ent_id, ent_ood)

    return {
        "arm": adapter.name,
        "n_id": int(pred_id.n),
        "n_ood": int(pred_ood.n),

        "accuracy": accuracy(pred_id.probs, y_id),
        "nll": nll(pred_id.probs, y_id),
        "ece": ece_value,
        "ece_equal_mass": ece_em,
        "auroc_native": auroc_native,
        "auroc_entropy": auroc_entropy,
        "n_bins": n_bins,

        # per-sample arrays consumed by the plots
        "y_id": y_id,
        "probs_id": pred_id.probs,
        "probs_ood": pred_ood.probs,
        "ood_score_id": pred_id.ood_score,
        "ood_score_ood": pred_ood.ood_score,
        "entropy_id": ent_id,
        "entropy_ood": ent_ood,
        "bin_conf": bin_conf,
        "bin_acc": bin_acc,
        "bin_count": bin_count,
        "bin_edges": bin_edges,

        # provenance, so the report can state exactly what produced the numbers
        "extra": dict(pred_id.extra),
    }

## Section 4 - the training loop

In [12]:
# =============================================================================
# Section 4 - one training loop for both objectives
# =============================================================================
def _is_vbll_model(model: nn.Module) -> bool:
    return isinstance(model, VBLLModel)


def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                cfg: Optional[Dict[str, Any]] = None, tag: str = ""):
    """Unified training loop.

    * arm (a) and arm (d)'s base network -> nn.CrossEntropyLoss on raw logits
    * arms (b) and (c) -> the VBLL ELBO, differentiated through
      out.train_loss_fn(y), with out.val_loss_fn(y) for validation

    The VBLL head is excluded from weight decay. Its prior term already regularises
    it (and the ELBO is scaled by reg_weight = 1/N), so adding weight decay would
    double-count the prior.

    Early stopping is on validation accuracy; the best weights are restored before
    returning, so no arm is ever evaluated mid-training.
    """
    cfg = CFG if cfg is None else cfg
    is_vbll = _is_vbll_model(model)

    head_params = list(model.head.parameters()) if is_vbll else []
    head_ids = {id(p) for p in head_params}
    trunk_params = [p for p in model.parameters() if id(p) not in head_ids]

    groups = [{"params": trunk_params, "weight_decay": cfg["weight_decay"]}]
    if head_params:
        groups.append({"params": head_params, "weight_decay": 0.0})
    opt = torch.optim.AdamW(groups, lr=cfg["lr"])

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best = {"val_acc": -1.0, "epoch": -1, "state": None}
    patience_left = cfg["early_stop_patience"]

    for epoch in range(1, cfg["epochs"] + 1):
        t0 = time.time()

        # ---------------- train ----------------
        model.train()
        run_loss, n_seen = 0.0, 0
        for x, y in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            if is_vbll:
                out = model(x)
                loss = out.train_loss_fn(y)          # the negative ELBO
            else:
                loss = F.cross_entropy(model(x), y)

            loss.backward()
            opt.step()

            run_loss += float(loss.detach()) * x.size(0)
            n_seen += x.size(0)
        train_loss = run_loss / max(n_seen, 1)

        # ---------------- validate ----------------
        model.eval()
        val_loss, correct, total = 0.0, 0, 0

        # For VBLL the predictive is stochastic, so validation accuracy would wobble
        # between epochs and early stopping would fire on MC noise rather than on a
        # real change. Pinning the RNG for the validation pass only makes the
        # epoch-to-epoch comparison meaningful without perturbing training.
        _mc_ctx = deterministic_mc(cfg["seed"]) if is_vbll else nullcontext()
        with _mc_ctx:
            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(DEVICE, non_blocking=True)
                    y = y.to(DEVICE, non_blocking=True)
                    if is_vbll:
                        out = model(x)
                        val_loss += float(out.val_loss_fn(y).detach()) * x.size(0)
                        pred = out.predictive.probs.argmax(dim=-1)
                    else:
                        z = model(x)
                        val_loss += float(F.cross_entropy(z, y).detach()) * x.size(0)
                        pred = z.argmax(dim=-1)
                    correct += int((pred == y).sum())
                    total += x.size(0)
        val_loss /= max(total, 1)
        val_acc = correct / max(total, 1)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        improved = val_acc > best["val_acc"]
        if improved:
            best = {
                "val_acc": val_acc,
                "epoch": epoch,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            }
            patience_left = cfg["early_stop_patience"]
        else:
            patience_left -= 1

        print("  [{:<15}] epoch {:>3d}/{}  train_loss={:9.4f}  val_loss={:9.4f}  "
              "val_acc={:.4f}  ({:.1f}s){}".format(
                  tag, epoch, cfg["epochs"], train_loss, val_loss, val_acc,
                  time.time() - t0, "  *" if improved else ""))

        if patience_left <= 0:
            print("  [{:<15}] early stop at epoch {} (best epoch {})".format(
                tag, epoch, best["epoch"]))
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"])
    return history, best

## Section 6 - the ablation study

Six arms, one factor changed at a time, all on top of the same backbone and the same
training loop as the main run. Three questions are being asked:

1. **How much does the covariance parameterisation matter?** - `diagonal` vs `lowrank` vs `dense`.
2. **How much does the prior scale matter?** - `prior_scale` 0.1 vs 10.0 against the default 1.0.
3. **What happens when the head is built without `return_ood`?** - the adapter is supposed
   to fail loudly and by name.

### Three things to know before running

**1. This cell is resumable.** Each finished arm is appended to `results_ablations.csv`
immediately, and an arm already present in that file is skipped. A Colab recycle or a timeout
therefore costs only the arm in flight, not the whole sweep. If the runtime dies, just run the
same cell again and it continues from where it stopped.

**2. `disc_dense` is placed last on purpose - it is not a memory risk, it is a *compute* risk.**
Its covariance is a full `(10, 256, 256)` block per class, so the Jensen bound needs a 256x256
Cholesky plus triangular solves at every step. The extra parameters are only about 2.6 MB
(no OOM risk on a 15 GB T4), but the step is roughly 375x slower than `diagonal` in a CPU
measurement. Expect it to dominate the wall-clock time of the whole notebook.

**3. One row is not comparable to the others.** `disc_no_ood` records the **validation**
accuracy (there is no test-set evaluation, because evaluation is exactly what is supposed to
fail). Every other row records the **test** accuracy. Do not rank `disc_no_ood`'s accuracy
against the other rows - its only meaning is "did the adapter raise, by name, as designed".

In [13]:
# =============================================================================
# Section 6 - ablations, RESUMABLE
# =============================================================================
# One factor at a time, on top of the same backbone and the same training loop as the main
# run. Each finished arm is written to results_ablations.csv immediately, and an arm that is
# already in that file is skipped, so a Colab recycle or a timeout costs only the arm in
# flight - not the whole sweep.
#
# Ordering: 'disc_dense' is last on purpose. Its covariance is a full 256x256 matrix per
# class, so its Jensen bound needs a 256x256 Cholesky and triangular solves at every step.
# It is not a memory risk (a few MB of extra parameters) but it is by far the most expensive
# arm, so it should not block the other five.
import os
import numpy as np
import pandas as pd

ABLATIONS = [
    ("disc_diagonal",  dict(kind="disc", parameterization="diagonal")),
    ("disc_lowrank",   dict(kind="disc", parameterization="lowrank")),
    ("disc_prior_0.1", dict(kind="disc", prior_scale=0.1)),
    ("disc_prior_10",  dict(kind="disc", prior_scale=10.0)),
    ("disc_no_ood",    dict(kind="disc", return_ood=False)),
    ("disc_dense",     dict(kind="disc", parameterization="dense")),
]

ABL_CSV = os.path.join(CFG["out_dir"], "results_ablations.csv")
COLS = ["ablation", "accuracy", "nll", "ece", "auroc_native", "note"]

done = {}
if os.path.exists(ABL_CSV):
    for _, r in pd.read_csv(ABL_CSV).iterrows():
        done[r["ablation"]] = r.to_dict()
    print("resuming - already finished:", sorted(done))
else:
    print("starting fresh")


def _save_abl():
    pd.DataFrame([done[t] for t, _ in ABLATIONS if t in done],
                 columns=COLS).to_csv(ABL_CSV, index=False)


for tag, kw in ABLATIONS:
    if tag in done:
        print("skip {} (already done)".format(tag))
        continue

    print("=" * 74)
    print("ablation: {}".format(tag))
    print("=" * 74)

    set_seed(CFG["seed"])
    model_a = build_vbll_model(**kw)
    _, best_a = train_model(model_a, train_loader, val_loader, CFG, tag=tag)

    if kw.get("return_ood", True) is False:
        # This arm asserts a CONTRACT: with return_ood=False the head carries no ood_scores,
        # and the adapter must fail loudly and by name rather than raising a bare
        # AttributeError somewhere downstream. We keep the message as the evidence.
        try:
            VBLLAdapter(model_a, tag).predict(test_loader)
            note = "adapter did NOT raise - unexpected"
        except RuntimeError as err:
            note = "adapter raised as designed: " + str(err)[:70]
        row = dict(ablation=tag, accuracy=round(best_a["val_acc"], 4),
                   nll=np.nan, ece=np.nan, auroc_native=np.nan, note=note)
    else:
        r_a = evaluate_model(VBLLAdapter(model_a, tag), test_loader, ood_loader,
                             n_bins=CFG["n_bins"])
        row = dict(ablation=tag, accuracy=round(r_a["accuracy"], 4),
                   nll=round(r_a["nll"], 4), ece=round(r_a["ece"], 4),
                   auroc_native=round(r_a["auroc_native"], 4), note="")

    done[tag] = row
    _save_abl()          # durable: the row survives a disconnect
    print("  saved:", {k: v for k, v in row.items() if k != "note"})

print()
print(pd.DataFrame([done[t] for t, _ in ABLATIONS if t in done],
                   columns=COLS).to_string(index=False))

starting fresh
ablation: disc_diagonal
  [disc_diagonal  ] epoch   1/30  train_loss=   0.6493  val_loss=   0.1232  val_acc=0.9615  (28.8s)  *
  [disc_diagonal  ] epoch   2/30  train_loss=   0.3597  val_loss=   0.0963  val_acc=0.9678  (13.0s)  *
  [disc_diagonal  ] epoch   3/30  train_loss=   0.2821  val_loss=   0.0940  val_acc=0.9703  (12.6s)  *
  [disc_diagonal  ] epoch   4/30  train_loss=   0.2307  val_loss=   0.0805  val_acc=0.9757  (12.8s)  *
  [disc_diagonal  ] epoch   5/30  train_loss=   0.1969  val_loss=   0.0732  val_acc=0.9765  (12.8s)  *
  [disc_diagonal  ] epoch   6/30  train_loss=   0.1672  val_loss=   0.0701  val_acc=0.9772  (12.5s)  *
  [disc_diagonal  ] epoch   7/30  train_loss=   0.1415  val_loss=   0.0701  val_acc=0.9780  (12.6s)  *
  [disc_diagonal  ] epoch   8/30  train_loss=   0.1236  val_loss=   0.0670  val_acc=0.9798  (17.1s)  *
  [disc_diagonal  ] epoch   9/30  train_loss=   0.1056  val_loss=   0.0669  val_acc=0.9793  (12.5s)
  [disc_diagonal  ] epoch  10/30  tra

## Section 6 (continued) - from CSV to a Markdown table

Run this after the sweep finishes. The printed table can be pasted straight into the report.

In [14]:
# =============================================================================
# Section 6 - results_ablations.csv -> Markdown (paste straight into the report)
# =============================================================================
import os
import pandas as pd

df = pd.read_csv(os.path.join(CFG["out_dir"], "results_ablations.csv"))


def _fmt(v):
    # An empty note comes back from a CSV round-trip as NaN; render both as an em dash.
    if pd.isna(v) or (isinstance(v, str) and not v.strip()):
        return "—"
    if isinstance(v, float):
        return "%.4f" % v
    return str(v)


cols = list(df.columns)
print("| " + " | ".join(cols) + " |")
print("|" + "|".join(["---"] * len(cols)) + "|")
for _, row in df.iterrows():
    print("| " + " | ".join(_fmt(row[c]) for c in cols) + " |")

| ablation | accuracy | nll | ece | auroc_native | note |
|---|---|---|---|---|---|
| disc_diagonal | 0.9839 | 0.0571 | 0.0061 | 0.9208 | — |
| disc_lowrank | 0.9838 | 0.0615 | 0.0059 | 0.9390 | — |
| disc_prior_0.1 | 0.9833 | 0.0574 | 0.0040 | 0.9103 | — |
| disc_prior_10 | 0.9832 | 0.0610 | 0.0077 | 0.9145 | — |
| disc_no_ood | 0.9850 | — | — | — | adapter raised as designed: [disc_no_ood] the VBLL head was built with return_ood=False, so out.oo |
| disc_dense | 0.9835 | 0.0584 | 0.0056 | 0.9253 | — |
